In [ ]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_excel_filename, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import math
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
import random
from scipy.optimize import minimize
from scipy.stats import false_discovery_control, kendalltau, linregress, pearsonr, spearmanr, ttest_ind, wilcoxon
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from stock_screener import check_conds_tech, check_conds_fund, EM_rating, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

# Connect to TradingView
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
# --- Select Stock Universe from Excel File ---
# Load pre-screened stocks from the stock screener output file
# Filter for large-cap stocks (market cap > $10B) to ensure liquidity
def extract_stocks(current_date, index_name, index_dict, period=252, RS=90, all_stocks=True, cap_threshold=10):
    excel_filename = get_excel_filename(current_date, index_name, index_dict, period, RS, all_stocks)
    excel_df = pd.read_excel(excel_filename)
    stocks_df = excel_df.loc[excel_df["Market Cap (B, USD)"] > cap_threshold, ["Stock", "Market Cap (B, USD)"]]
    return stocks_df["Stock"].tolist()

# --- Fetch Price Data ---
# Fetch historical closing prices for all stocks
def fetch_price_data(stocks, current_date, benchmark="^GSPC"):
    # Include S&P 500 in the calculation
    all_tickers = stocks + [benchmark]

    # Fetch historical price data for all stocks
    price_data = {}
    for stock in tqdm(all_tickers, desc="Fetching price data"):
        df = get_df(stock, current_date)
        df = df[df.index <= current_date]
        price_data[stock] = df["Close"]
    return price_data

# --- Weekly Returns Z-Score Analysis ---
# Z-score measures how unusual this week's return is vs historical weekly returns
# High z-score (>2) = unusually strong week, may indicate temporary spike
def compute_weekly_return_zscores(price_data, stocks, period_week_zscore, days_per_week):
    df_prices_weekly = pd.DataFrame({stock: price_data[stock].tail(period_week_zscore + days_per_week) for stock in stocks})
    weekly_prices = df_prices_weekly.iloc[::days_per_week]
    weekly_returns = weekly_prices.pct_change(fill_method=None).dropna()

    mean_return = weekly_returns.mean()
    std_return = weekly_returns.std()
    recent_return = weekly_returns.iloc[-1]
    z_scores = (recent_return - mean_return) / std_return

    return weekly_returns, mean_return, std_return, recent_return, z_scores

# --- Momentum and Volatility Analysis ---
# Momentum: price ratio (recent / year-ago) - measures trend strength over 1 year
# Volatility: standard deviation of daily returns - measures price variability
# Vol-adj Momentum: momentum / volatility - risk-adjusted trend
def calculate_momentum_volatility(price_data, stocks, period_mom_short, period_mom_long, period_vol, benchmark="^GSPC"):
    """
    Calculates Total Momentum, Idiosyncratic Momentum, and Volatility.
    
    Parameters:
    - price_data: DataFrame with columns as tickers and rows as dates ("Close").
    - stocks: List of stock tickers to analyze.
    - period_mom_short: The "recent" exclusion period (e.g., 21 for 1 month).
    - period_mom_long: The lookback period (e.g., 252 for 1 year).
    - period_vol: Lookback for volatility (e.g., 60).
    - benchmark: Ticker for the market index (e.g., ^GSPC).
    
    Returns:
    - momentum_list, idio_momentum_list, volatility_list
    """
    # Include S&P 500 in the calculation
    all_tickers = stocks + [benchmark]
    
    # Build DataFrame once for vectorized operations
    df_prices = pd.DataFrame({stock: price_data[stock].tail(period_mom_long + 1) for stock in all_tickers})

    # Vectorized calculation of daily returns
    daily_returns = df_prices.pct_change(fill_method=None).dropna()

    # Vectorized volatility calculation: std of daily returns over period_vol
    volatility_series = daily_returns[stocks].tail(period_vol).std()
    
    # Vectorized calculation of total momentum
    momentum_series = df_prices[stocks].iloc[- period_mom_short] / df_prices[stocks].iloc[- period_mom_long] - 1
    
    # Vectorized beta calculation
    # We calculate the covariance matrix of ALL stocks + Benchmark
    # Then we select only the column corresponding to the benchmark
    cov_matrix = daily_returns.cov()
    cov_with_benchmark = cov_matrix[benchmark].loc[stocks] # Filter for just our stocks
    beta_series = cov_with_benchmark / daily_returns[benchmark].var()
    
    # Calculate idiosyncratic momentum
    # Formula: r_idio = r_stock - (beta * r_market)
    
    # Calculate benchmark return over the same specific window
    benchmark_return = df_prices[benchmark].iloc[- period_mom_short] / df_prices[benchmark].iloc[- period_mom_long] - 1
    
    # Calculate residuals
    idio_momentum_series = momentum_series - (beta_series * benchmark_return)

    # Convert series to lists
    return momentum_series.tolist(), idio_momentum_series.tolist(), volatility_series.tolist()

# --- Hierarchical Clustering Analysis ---
# Groups stocks by return correlation using Ward's linkage (minimizes within-cluster variance)
def hierarchical_clustering(price_data, stocks, period_pca, num_clusters):
    # Ensure we have at least 2 stocks to cluster
    if len(stocks) < 2:
        return None, np.array([1] * len(stocks))

    # Build the price dataframe
    df_prices_cluster = pd.DataFrame({stock: price_data[stock].tail(period_pca) for stock in stocks})
    
    # Calculate returns
    returns = df_prices_cluster.pct_change(fill_method=None)
    
    # Check if we have enough valid rows to calculate correlation
    # We need at least 2 rows of returns to have a correlation
    if len(returns.dropna(how="all")) < 2:
        print(f"Warning: Not enough data for clustering in this period. Skipping...")
        return None, np.array([1] * len(stocks))

    # Correlation Matrix
    corr_matrix = returns.corr()
    
    # Handle NaNs in the correlation matrix (caused by zero volatility/flat prices)
    corr_matrix = corr_matrix.fillna(0)
    
    # Convert to distance matrix
    dist_matrix = np.sqrt(2 * (1 - corr_matrix))
    
    # Hierarchical clustering
    try:
        # squareform() expects a symmetric matrix with 0s on the diagonal
        condensed_dist = squareform(dist_matrix, checks=False)
        
        if condensed_dist.size == 0:
            raise ValueError("Condensed distance matrix is empty.")
            
        Z = linkage(condensed_dist, method="ward")
        cluster_labels = fcluster(Z, t=num_clusters, criterion="maxclust")
        return Z, cluster_labels
        
    except Exception as e:
        print(f"Clustering failed: {e}")
        # Fallback: Assign everyone to Cluster 1
        return None, np.array([1] * len(stocks))

# --- Combine All Metrics into Single DataFrame ---
def create_combined_dataframe(stocks, cluster_labels, momentum_list, idio_momentum_list, volatility_list, 
                               mean_return, std_return, recent_return, z_scores):
    """
    Consolidate all computed metrics into a single DataFrame for analysis.
    Adds vol-adjusted momentum and ranks stocks by this metric.
    """
    df_combined = pd.DataFrame({
        "Stock": stocks,
        "Cluster": cluster_labels,
        "Momentum": momentum_list,
        "Idio Momentum": idio_momentum_list,
        "Volatility": volatility_list,
        "Mean Weekly Return (%)": (mean_return * 100).values,
        "Std Weekly Return (%)": (std_return * 100).values,
        "This Week Return (%)": (recent_return * 100).values,
        "Z-Score": z_scores.values
    })
    
    # Vol-adjusted momentum: normalize momentum by volatility for risk-adjusted comparison
    df_combined["Vol-adj Momentum"] = df_combined["Momentum"] / df_combined["Volatility"]
    df_combined["Vol-adj Idio Momentum"] = df_combined["Idio Momentum"] / df_combined["Volatility"]
    
    # Sort by vol-adjusted momentum (descending) and create rank index starting from 1
    df_combined = df_combined.sort_values("Vol-adj Momentum", ascending=False).reset_index(drop=True)
    df_combined.index = df_combined.index + 1
    df_combined.index.name = "Rank"
    return df_combined

# --- Dendrogram Visualization ---
def plot_dendrogram(Z, stocks, period_pca):
    """
    Visualize hierarchical clustering structure as a dendrogram.
    Y-axis (distance) shows how dissimilar clusters are when merged.
    """
    plt.figure(figsize=(14, 8))
    dendrogram(Z, labels=stocks, leaf_rotation=90, leaf_font_size=8)
    plt.title(f"Hierarchical Clustering of {len(stocks)} Stocks (Past {period_pca} Days)")
    plt.ylabel("Distance")
    plt.tight_layout()
    plt.show()

# --- Filter and Display Stocks by Cluster ---
# Exclude stocks with z-score > 2 (unusually high recent returns)
# These may be experiencing unsustainable spikes rather than steady momentum
def filter_stocks_by_zscore(df_combined, z_score_threshold=2):
    """Filter stocks and display by cluster."""
    df_filtered = df_combined[df_combined["Z-Score"] <= z_score_threshold]

    print(f"Stocks by Cluster (Ranked by Vol-adj Momentum, Z-Score <= {z_score_threshold}):")
    for cluster_id in sorted(df_filtered["Cluster"].unique()):
        cluster_df = df_filtered[df_filtered["Cluster"] == cluster_id]
        cluster_stocks = cluster_df["Stock"].tolist()
        print(f"Cluster {cluster_id} ({len(cluster_df)} stocks): {', '.join(cluster_stocks)}")
    
    return df_filtered

def select_top_stocks_with_weights(df_combined, num_stocks, use_hca=True, cluster_cap=0.4):
    """
    Select top stocks using HCA cluster-aware selection or simple momentum.
    """
    if df_combined.empty:
        return pd.DataFrame()
    
    # Ensure the dataframe is sorted by momentum before ranking
    df_copy = df_combined.sort_values("Vol-adj Momentum", ascending=False).copy()

    # Assign intra-cluster rank
    df_copy["Intra-cluster Rank"] = df_copy.groupby("Cluster").cumcount()

    if use_hca:
        # Sort: First by intra-cluster rank, then by the original sort order (Vol-adj Momentum)
        top_stocks = df_copy.sort_values(by=["Intra-cluster Rank", "Vol-adj Momentum"], ascending=[True, False]).head(num_stocks)

    else:
        if cluster_cap is not None:
            # Calculate the dynamic limit per cluster
            max_per_cluster = math.ceil(num_stocks * cluster_cap)

            # Simple momentum selection
            top_stocks = df_copy[df_copy["Intra-cluster Rank"] < max_per_cluster].head(num_stocks)
        else:
            # Simple momentum selection without cluster cap
            top_stocks = df_copy.head(num_stocks)
    
    volatilities = top_stocks["Volatility"].values
    zscores = top_stocks["Z-Score"].values
    inv_vol = 1 / volatilities
    weights = inv_vol / inv_vol.sum()

    return pd.DataFrame({
        "Cluster": top_stocks["Cluster"],
        "Stock": top_stocks["Stock"],
        "Volatility": volatilities,
        "Vol-adj Momentum": top_stocks["Vol-adj Momentum"],
        "Vol-adj Idio Momentum": top_stocks["Vol-adj Idio Momentum"],
        "Z-Score": zscores,
        "Weight (%)": weights * 100,
    })

In [ ]:
# --- Main Monthly Portfolio Construction Function ---
def construct_month_portfolio(current_date, index_name, index_dict, parameters, use_hca=True, cluster_cap=0.4):
    """
    Construct a monthly momentum portfolio based on hierarchical clustering and vol-adjusted momentum.
    """
    # Extract stocks
    stocks = extract_stocks(current_date, index_name, index_dict)

    # Check if we have enough stocks
    if len(stocks) < 1:
        print("Not enough stocks to construct a portfolio.")
        return pd.DataFrame(), 0
    
    # Fetch price data
    price_data = fetch_price_data(stocks, current_date)
    
    # Weekly returns z-score analysis
    weekly_returns, mean_return, std_return, recent_return, z_scores = compute_weekly_return_zscores(
        price_data, stocks, parameters["period_week_zscore"], parameters["days_per_week"])
    
    # Momentum and volatility calculation
    momentum_list, idio_momentum_list, volatility_list = calculate_momentum_volatility(
        price_data, stocks, parameters["period_mom_short"], parameters["period_mom_long"], parameters["period_vol"])
    
    # Hierarchical clustering
    Z, cluster_labels = hierarchical_clustering(price_data, stocks, parameters["period_pca"], parameters["num_clusters"])
    
    # Create combined DataFrame
    df_combined = create_combined_dataframe(
        stocks, cluster_labels, momentum_list, idio_momentum_list, volatility_list,
        mean_return, std_return, recent_return, z_scores)
    
    # Filter stocks by z-score
    df_filtered = filter_stocks_by_zscore(df_combined)

    # Extract number of stocks after filtering
    num_stocks_filtered = len(df_filtered)
    print(f"Number of stocks after Z-Score filtering: {num_stocks_filtered}")
    
    # Select top stocks with weights
    portfolio_df = select_top_stocks_with_weights(df_filtered, num_stocks=parameters["num_stocks"], use_hca=use_hca, cluster_cap=cluster_cap)
    
    return portfolio_df, num_stocks_filtered

In [ ]:
# --- Parameters ---
days_per_week = 5  # Trading days in a week
num_weeks = 52  # Weeks in a year
period_week_zscore = days_per_week * num_weeks  # 1-year lookback for weekly return z-score
period_pca = 126  # 6-month lookback for PCA clustering
period_mom_short = 21  # 1-month lookback for short-term momentum
period_mom_long = 252  # 1-year lookback for long-term momentum
period_vol = 60  # 3-month volatility window
num_clusters = 5  # Number of clusters for hierarchical grouping
num_stocks = 5  # Number of stocks to select for the portfolio
parameters = {
    "days_per_week": days_per_week,
    "period_week_zscore": period_week_zscore,
    "period_pca": period_pca,
    "period_mom_short": period_mom_short,
    "period_mom_long": period_mom_long,
    "period_vol": period_vol,
    "num_clusters": num_clusters,
    "num_stocks": num_stocks
}

# --- Extract Stocks ---
stocks = extract_stocks("2026-01-10", index_name, index_dict)

# --- Fetch Price Data ---
price_data = fetch_price_data(stocks, "2026-01-10")

# --- Compute Weekly Return Z-Scores ---
weekly_returns, mean_return, std_return, recent_return, z_scores = compute_weekly_return_zscores(
    price_data, stocks, period_week_zscore, days_per_week
)

# --- Calculate Momentum and Volatility ---
momentum_list, idio_momentum_list, volatility_list = calculate_momentum_volatility(price_data, stocks, period_mom_short, period_mom_long, period_vol)

# --- Hierarchical Clustering Analysis ---
Z, cluster_labels = hierarchical_clustering(price_data, stocks, period_pca, num_clusters)

# --- Create Combined DataFrame ---
df_combined = create_combined_dataframe(
    stocks, cluster_labels, momentum_list, idio_momentum_list, volatility_list,
    mean_return, std_return, recent_return, z_scores
)

# --- Display Results ---
pd.set_option("display.max_rows", None)
display(df_combined)
plot_dendrogram(Z, stocks, period_pca)

# --- Filter Stocks by Z-Score and Display by Cluster ---
df_filtered = filter_stocks_by_zscore(df_combined)
portfolio_df = select_top_stocks_with_weights(df_filtered, num_stocks, use_hca=False)
num_stocks_filtered = len(df_filtered)
print(f"Number of stocks after Z-Score filtering: {num_stocks_filtered}")
display(portfolio_df)

In [ ]:
# --- Rebalance dates for monthly portfolio analysis ---
rebalance_dates = [
    "2024-02-01", "2024-03-01", "2024-04-01", "2024-05-01", "2024-05-31",
    "2024-07-01", "2024-08-03", "2024-08-31", "2024-10-03", "2024-11-02", "2024-11-30",
    "2025-01-04", "2025-02-01", "2025-03-01", "2025-04-05", "2025-05-01", "2025-06-03",
    "2025-07-01", "2025-08-01", "2025-09-06", "2025-10-01", "2025-11-01", "2025-12-04", 
    "2026-01-03"
]

In [ ]:
# --- Construct Monthly Portfolios ---
monthly_portfolios_hca = {}
monthly_portfolios_nohca_cap40 = {} # No HCA with 40% cluster cap
monthly_portfolios_nohca_nocap = {} # No HCA without cluster cap
num_stocks_filtered_list = []
for date in rebalance_dates:
    # With Hierarchical Clustering Analysis
    portfolio_df, num_stocks_filtered = construct_month_portfolio(date, index_name, index_dict, parameters)
    monthly_portfolios_hca[date] = portfolio_df
    num_stocks_filtered_list.append(num_stocks_filtered)
    print(f"\nPortfolio for {date} with HCA:\n")
    display(portfolio_df)
    # Without Hierarchical Clustering Analysis (40% cluster cap)
    portfolio_df, num_stocks_filtered = construct_month_portfolio(date, index_name, index_dict, parameters, use_hca=False)
    monthly_portfolios_nohca_cap40[date] = portfolio_df
    print(f"\nPortfolio for {date} without HCA (40% cluster cap):\n")
    display(portfolio_df)
    # Without Hierarchical Clustering Analysis (no cluster cap)
    portfolio_df, num_stocks_filtered = construct_month_portfolio(date, index_name, index_dict, parameters, use_hca=False, cluster_cap=None)
    monthly_portfolios_nohca_nocap[date] = portfolio_df
    print(f"\nPortfolio for {date} without HCA (no cluster cap):\n")
    display(portfolio_df)

In [ ]:
# --- Configuration & Initialization ---
# Set the base value to 100 for percentage tracking and define the 0.1% fee
current_capital_hca = 100.0
current_capital_nohca_cap40 = 100.0
current_capital_nohca_nocap = 100.0
transaction_fee_rate = 0.001 

# Risk-free rate assumptions
annual_rf_rate = 0.03 # Assuming 3% annual return on cash
daily_rf_return = (1 + annual_rf_rate) ** (1/252) # Convert annual to daily multiplier

mmfi_exposure = 0.3 # MMFI-triggered exposure level
buy_tqqq = False

equity_curve_hca = []
equity_curve_nohca_cap40 = []
equity_curve_nohca_nocap = []
equity_dates = []
exposure_curve = [] # Track weighting here

# Retrieve the full trading calendar using the S&P 500 as a reference
spmo_df = get_df("SPMO", current_date)
sp500_df = get_df("^GSPC", current_date)
sp500_df["SMA 200"] = SMA(sp500_df, 200)
tqqq_df = get_df("TQQQ", current_date)
tqqq_df["Daily Return"] = tqqq_df["Close"] / tqqq_df["Close"].shift(1)
mmth_df = get_df("MMTH", current_date, method="tradingview")
mmfi_df = get_df("MMFI", current_date, method="tradingview")
trading_dates = sp500_df.index

# --- Portfolio Backtest Loop ---
for i in range(len(rebalance_dates) - 1):
    rebalance_date = rebalance_dates[i]
    next_rebalance_date = rebalance_dates[i + 1]
    signal_date = trading_dates[trading_dates <= rebalance_date][-1]
    
    try:
        buy_date = trading_dates[trading_dates > rebalance_date][0]
        sell_date = trading_dates[trading_dates > next_rebalance_date][0]
    except IndexError:
        break 

    # Detemine exposure level
    current_sp_close = sp500_df["Close"].loc[signal_date]
    current_sp_sma = sp500_df["SMA 200"].loc[signal_date]
    current_mmth = mmth_df["Close"].loc[signal_date]
    base_exposure = 0.0 if ((current_sp_close < current_sp_sma) or (current_mmth < 50)) else 1.0

    # Determine the trading period
    period_dates = trading_dates[(trading_dates >= buy_date) & (trading_dates < sell_date)]

    # Pre-calculate daily returns for BOTH portfolios
    def get_period_returns(portfolio, period_dates, buy_date, sell_date):
        # Check if portfolio is empty or None
        if portfolio is None or portfolio.empty:
            # FALLBACK: Use S&P 500 returns
            prices = sp500_df.loc[period_dates, "Close"]
            first_day_open = sp500_df.loc[buy_date, "Open"]
            
            first_day_ret = prices.iloc[0] / first_day_open
            other_days_ret = prices / prices.shift(1)
            other_days_ret.iloc[0] = first_day_ret
            return other_days_ret
        
        # NORMAL LOGIC: Calculate weighted stock returns
        rets_list = []
        for _, row in portfolio.iterrows():
            ticker = row["Stock"]
            weight = row["Weight (%)"] / 100
            sell_date_str = sell_date.strftime("%Y-%m-%d")
            stock_df = get_df(ticker, sell_date_str)
            if stock_df is not None and buy_date in stock_df.index:
                prices = stock_df.loc[period_dates, "Close"].reindex(period_dates)
                # Handle execution: Buy at Open of buy_date
                first_day_return = prices.iloc[0] / stock_df.loc[buy_date, "Open"]
                other_days_return = prices / prices.shift(1)
                other_days_return.iloc[0] = first_day_return
                rets_list.append(other_days_return * weight)
            else:
                # If a specific stock is missing data, that weight stays in cash (1.0)
                rets_list.append(pd.Series(1.0 * weight, index=period_dates))
        return pd.concat(rets_list, axis=1).sum(axis=1)

    daily_rets_hca = get_period_returns(monthly_portfolios_hca[rebalance_date], period_dates, buy_date, sell_date)
    daily_rets_nohca_cap40 = get_period_returns(monthly_portfolios_nohca_cap40[rebalance_date], period_dates, buy_date, sell_date)
    daily_rets_nohca_nocap = get_period_returns(monthly_portfolios_nohca_nocap[rebalance_date], period_dates, buy_date, sell_date)

    # --- DAILY EXECUTION LOOP (Handles MMFI Trigger) ---
    active_exposure = base_exposure
    mmfi_triggered = False
    
    # Apply initial fees to both
    if active_exposure > 0:
        current_capital_hca *= (1 - (transaction_fee_rate * active_exposure))
        current_capital_nohca_cap40 *= (1 - (transaction_fee_rate * active_exposure))
        current_capital_nohca_nocap *= (1 - (transaction_fee_rate * active_exposure))

    for date in period_dates:
        if buy_tqqq and mmfi_triggered:
            # Calculate today's growth based on the exposure decided YESTERDAY
            growth_hca = growth_nohca_cap40 = growth_nohca_nocap = (tqqq_df["Daily Return"].loc[date] * active_exposure) + (daily_rf_return * (1 - active_exposure))
        else:
            # Calculate today's growth based on the exposure decided YESTERDAY
            growth_hca = (daily_rets_hca.loc[date] * active_exposure) + (daily_rf_return * (1 - active_exposure))
            growth_nohca_cap40 = (daily_rets_nohca_cap40.loc[date] * active_exposure) + (daily_rf_return * (1 - active_exposure))
            growth_nohca_nocap = (daily_rets_nohca_nocap.loc[date] * active_exposure) + (daily_rf_return * (1 - active_exposure))
        
        # Update capital and equity curves
        current_capital_hca *= growth_hca
        current_capital_nohca_cap40 *= growth_nohca_cap40
        current_capital_nohca_nocap *= growth_nohca_nocap
        equity_curve_hca.append(current_capital_hca)
        equity_curve_nohca_cap40.append(current_capital_nohca_cap40)
        equity_curve_nohca_nocap.append(current_capital_nohca_nocap)
        equity_dates.append(date)
        exposure_curve.append(active_exposure) # Record current weight

        # Check for MMFI trigger TODAY to affect TOMORROW'S open
        current_mmfi = mmfi_df["Close"].loc[date]
        
        if not mmfi_triggered and current_mmfi < 10:
            mmfi_triggered = True # Override the logic for the rest of this rebalance period

            # Fee for changing exposure
            trade_size = abs(active_exposure - mmfi_exposure)
            if trade_size > 0:
                fee = (1 - transaction_fee_rate * trade_size)
                current_capital_hca *= fee
                current_capital_nohca_cap40 *= fee
                current_capital_nohca_nocap *= fee
            
            active_exposure = mmfi_exposure
            # Note: This active_exposure will be used starting the NEXT date in the loop

# --- Benchmark Processing ---
# Create the portfolio series
strategy_series_hca = pd.Series(equity_curve_hca, index=equity_dates, name="Strategy (HCA)")
strategy_series_nohca_cap40 = pd.Series(equity_curve_nohca_cap40, index=equity_dates, name="Strategy (No HCA, 40% Cap)")
strategy_series_nohca_nocap = pd.Series(equity_curve_nohca_nocap, index=equity_dates, name="Strategy (No HCA, No Cap)")

# Normalize strategy to 100 at the start so it aligns with the benchmark
strategy_series_hca = (strategy_series_hca / strategy_series_hca.iloc[0]) * 100
strategy_series_nohca_cap40 = (strategy_series_nohca_cap40 / strategy_series_nohca_cap40.iloc[0]) * 100
strategy_series_nohca_nocap = (strategy_series_nohca_nocap / strategy_series_nohca_nocap.iloc[0]) * 100
# Create the exposure series
exposure_series = pd.Series(exposure_curve, index=equity_dates)

# Align S&P 500: Filter for the same dates and normalize to 100 at the start
sp500_series = sp500_df.loc[strategy_series_hca.index, "Close"]
sp500_series = (sp500_series / sp500_series.iloc[0]) * 100
sp500_series.name = "SP500"
spmo_series = spmo_df.loc[strategy_series_hca.index, "Close"]
spmo_series = (spmo_series / spmo_series.iloc[0]) * 100
spmo_series.name = "SPMO"

# Calculate drawdowns
def get_drawdown(series):
    rolling_max = series.cummax()
    return (series - rolling_max) / rolling_max

strat_dd_hca = get_drawdown(strategy_series_hca)
strat_dd_nohca_cap40 = get_drawdown(strategy_series_nohca_cap40)
strat_dd_nohca_nocap = get_drawdown(strategy_series_nohca_nocap)
sp500_dd = get_drawdown(sp500_series)
spmo_dd = get_drawdown(spmo_series)

# --- Performance Metrics Engine ---
def calculate_metrics(series, annual_rf_rate=0.03):
    # Calculate daily returns for volatility and risk-adjusted ratios
    returns = series.pct_change().dropna()
    
    # Annualized Return (CAGR)
    total_return = (series.iloc[-1] / series.iloc[0]) - 1
    years = (series.index[-1] - series.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1
    
    # Annualized Volatility
    vol = returns.std() * np.sqrt(252)
    
    # Sharpe Ratio
    sharpe = (cagr - annual_rf_rate) / vol

    # Sortino Ratio (downside deviation only)
    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252)
    sortino = (returns.mean() * 252) / downside_vol
    
    # Maximum Drawdown
    rolling_max = series.cummax()
    drawdowns = (series - rolling_max) / rolling_max
    max_dd = drawdowns.min()
    
    # Calmar Ratio
    calmar = cagr / abs(max_dd)
    
    return {
        "CAGR": f"{cagr:.2%}",
        "Sharpe": f"{sharpe:.2f}",
        "Sortino": f"{sortino:.2f}",
        "Calmar": f"{calmar:.2f}",
        "Annualized Vol": f"{vol:.2%}",
        "Max DD": f"{max_dd:.2%}"
    }

# Calculate metrics for both
strat_metrics_hca = calculate_metrics(strategy_series_hca)
strat_metrics_nohca_cap40 = calculate_metrics(strategy_series_nohca_cap40)
strat_metrics_nohca_nocap = calculate_metrics(strategy_series_nohca_nocap)
sp500_metrics = calculate_metrics(sp500_series)
spmo_metrics = calculate_metrics(spmo_series)

# --- Visualization & Comparison ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True, gridspec_kw={"height_ratios": [3, 1, 1]})

# Equity curve
ax1.plot(strategy_series_hca, label=f"Strategy (HCA) (Final: {strategy_series_hca.iloc[-1]:.2f})", color="blue", lw=1.5)
ax1.plot(strategy_series_nohca_cap40, label=f"Strategy (No HCA, 40% Cap) (Final: {strategy_series_nohca_cap40.iloc[-1]:.2f})", color="green", lw=1.5)
ax1.plot(strategy_series_nohca_nocap, label=f"Strategy (No HCA, No Cap) (Final: {strategy_series_nohca_nocap.iloc[-1]:.2f})", color="purple", lw=1.5)
ax1.plot(sp500_series, label=f"S&P 500 (Final: {sp500_series.iloc[-1]:.2f})", color="orange", lw=1.5, ls="--")
ax1.plot(spmo_series, label=f"SPMO (Final: {spmo_series.iloc[-1]:.2f})", color="red", lw=1.5, ls="--")
ax1.set_title("Momentum Strategy Performance", fontsize=16)
ax1.set_ylabel("Portfolio Value")
ax1.legend(loc="upper left")
ax1.grid(axis="y", alpha=0.2)
ax1.set_xlim(left=strategy_series_hca.index[0])

# Stock exposure
ax2.fill_between(exposure_series.index, exposure_series, color="royalblue", alpha=0.3, label="Stock Exposure")
ax2.plot(exposure_series, color="royalblue", lw=1)
ax2.set_ylabel("Stock Exposure")
ax2.grid(True, alpha=0.2)

# Drawdown
ax3.plot(strat_dd_hca, color="blue", lw=1, label="Strategy (HCA) Drawdown")
ax3.fill_between(strat_dd_hca.index, strat_dd_hca, 0, color="royalblue", alpha=0.2)
ax3.plot(strat_dd_nohca_cap40, color="green", lw=1, label="Strategy (No HCA, 40% Cap) Drawdown")
ax3.fill_between(strat_dd_nohca_cap40.index, strat_dd_nohca_cap40, 0, color="green", alpha=0.2)
ax3.plot(strat_dd_nohca_nocap, color="purple", lw=1, label="Strategy (No HCA, No Cap) Drawdown")
ax3.fill_between(strat_dd_nohca_nocap.index, strat_dd_nohca_nocap, 0, color="purple", alpha=0.2)
ax3.plot(sp500_dd, color="orange", lw=1, ls="--", label="S&P 500 Drawdown")
ax3.fill_between(sp500_dd.index, sp500_dd, 0, color="orange", alpha=0.2)
ax3.plot(spmo_dd, color="red", lw=1, ls="--", label="SPMO Drawdown")
ax3.fill_between(spmo_dd.index, spmo_dd, 0, color="red", alpha=0.2)
ax3.set_ylabel("Drawdown")
ax3.legend(loc="lower left", fontsize="small")
ax3.grid(True, alpha=0.2)

# Add grey vertical lines for rebalance dates
for idx, date in enumerate(pd.to_datetime(rebalance_dates)):
    if strategy_series_hca.index.min() <= date <= strategy_series_hca.index.max():
        ax1.axvline(x=date, color="grey", linestyle="--", linewidth=1, label="Rebalance" if idx == 0 else None)
# plt.savefig("Result/Figure/momentum_strategy_performance.png", dpi=300)
plt.tight_layout()
plt.show()

# -- Display Comparison Table ---
metrics_df = pd.DataFrame([strat_metrics_hca, strat_metrics_nohca_cap40, strat_metrics_nohca_nocap, sp500_metrics, spmo_metrics], index=["Strategy (HCA)", "Strategy (No HCA, 40% Cap)", "Strategy (No HCA, No Cap)", "S&P 500", "SPMO"])
print("--- Performance Comparison ---")
print(metrics_df.to_string())